# Review telomere insertion site plots

Interactively page through the zoomed-in insertion plots produced by
`src/telomererepeatloci/visualize_telomere_insertions.py`, across **every patient** under a
results root in one sitting. That script only ever writes a PDF per site
(`{prefix}{pid}_{chrom}_{center}.pdf` — the PNG output line is commented out/dead), at a
large 35x20-inch figure size, so this notebook rasterizes each PDF's first page and
downscales it for display. Mark each site Pass/Fail, add an optional note, and results
are written to a CSV as you go. A final cell joins your annotations back onto each
patient's own `*_extended_with_confidence_filtered.tsv` (the file with the final,
already-support-filtered insertion sites) and writes an `*_annotated.tsv` copy per
patient, plus a cross-patient pass-count summary.

Progress is resumable: re-running the notebook will skip plots already annotated in
the output CSV and resume where you left off.

In [ ]:
# One-time setup (uncomment and run if these aren't installed in your env)
# %pip install ipywidgets pymupdf pandas pillow

In [ ]:
from pathlib import Path

# --- CONFIGURE ME ---
# Parent directory holding one <sample>_TelomereRepeatLoci output directory per patient
# (each produced by a `telomere-repeat-loci` run), e.g. "results/".
RESULTS_ROOT = Path("results")

# Filename suffix of the final candidate-region table (the one with the already
# support-filtered insertion sites). Also used to derive each patient's PID from the
# filename -- the table itself carries no PID column, since it's one file per patient
# output dir.
CONFIDENCE_FILTERED_SUFFIX = (
    "_telomere_insertions_candidate_regions_extended_with_confidence_filtered.tsv"
)

ANNOTATIONS_CSV = (
    RESULTS_ROOT / "insertion_site_manual_annotations.csv"
)  # written to incrementally, one row per reviewed plot, across all patients
SUMMARY_CSV = (
    RESULTS_ROOT / "insertion_site_summary_per_pid.csv"
)  # per-pid pass counts, written by the last cell

# visualize_telomere_insertions.py only ever writes .pdf (the .png savefig call is commented
# out in that script); .png/.jpg are supported here too in case that line ever gets re-enabled.
PLOT_EXTENSIONS = (".pdf", ".png", ".jpg", ".jpeg")

# The source figures are rendered at 35x20 inches, so keep RENDER_DPI modest and let
# DISPLAY_MAX_WIDTH downscale for on-screen review -- full res would be slow and huge.
RENDER_DPI = 100
DISPLAY_MAX_WIDTH = 1400

In [ ]:
import re
import csv
from datetime import datetime, timezone

import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import io

# Filenames follow "{pid}_{chrom}_{center}.pdf", e.g. "H021-ABCD_chr7_123456789.pdf".
# pid itself may contain underscores/hyphens (patient/sample IDs commonly do), so parse
# from the right: the last two underscore-separated tokens are chrom and center, and
# everything before that is the pid.
FILENAME_RE = re.compile(r"^(?P<pid>.+)_(?P<chrom>[^_]+)_(?P<center>\d+)$")


def parse_plot_filename(path: Path):
    stem = path.stem
    m = FILENAME_RE.match(stem)
    if not m:
        return {"pid": stem, "chrom": None, "center": None}
    return {
        "pid": m.group("pid"),
        "chrom": m.group("chrom"),
        "center": int(m.group("center")),
    }


# Rendering can be slow: these PDFs are vector plots that can contain thousands of
# per-read rectangles (the source script caps at <3000 reads per track, plotted at
# 35x20 inches), so MuPDF has a lot of vector geometry to rasterize. We render once per
# file and cache the PNG on disk (keyed by source mtime) so re-viewing/resuming is instant.
CACHE_DIR = Path(".plot_render_cache")
CACHE_DIR.mkdir(exist_ok=True)


def render_plot_to_image(
    path: Path, dpi: int = RENDER_DPI, max_width: int = DISPLAY_MAX_WIDTH
) -> Image.Image:
    cache_path = CACHE_DIR / f"{path.stem}_{int(path.stat().st_mtime)}_{dpi}.png"
    if cache_path.exists():
        return Image.open(cache_path)

    if path.suffix.lower() == ".pdf":
        import fitz  # PyMuPDF

        doc = fitz.open(path)
        page = doc[0]
        zoom = dpi / 72
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
        img = Image.open(io.BytesIO(pix.tobytes("png")))
        doc.close()
    else:
        img = Image.open(path)

    if max_width and img.width > max_width:
        ratio = max_width / img.width
        img = img.resize((max_width, int(img.height * ratio)), Image.LANCZOS)

    img.save(cache_path)
    return img

In [ ]:
# Discover every patient's pipeline output dir under RESULTS_ROOT (anything with a
# candidate_region_tables/ subdir containing exactly one confidence-filtered table),
# then flatten all of their zoomed-in plots into one review queue.


class Patient:
    __slots__ = ("pid", "confidence_filtered_path", "plot_dir")

    def __init__(self, pid, confidence_filtered_path, plot_dir):
        self.pid = pid
        self.confidence_filtered_path = confidence_filtered_path
        self.plot_dir = plot_dir


patients = []
for patient_dir in sorted(p for p in RESULTS_ROOT.iterdir() if p.is_dir()):
    candidate_dir = patient_dir / "candidate_region_tables"
    if not candidate_dir.is_dir():
        continue

    matches = sorted(candidate_dir.glob(f"*{CONFIDENCE_FILTERED_SUFFIX}"))
    if not matches:
        print(f"Skipping {patient_dir.name}: no *{CONFIDENCE_FILTERED_SUFFIX} found")
        continue
    if len(matches) > 1:
        print(
            f"Skipping {patient_dir.name}: expected exactly one confidence-filtered "
            f"table, found {len(matches)}"
        )
        continue

    confidence_filtered_path = matches[0]
    pid = confidence_filtered_path.name[: -len(CONFIDENCE_FILTERED_SUFFIX)]
    patients.append(
        Patient(pid, confidence_filtered_path, patient_dir / "plots" / "zoomed_in")
    )

if not patients:
    raise FileNotFoundError(
        f"No patient output dirs with a *{CONFIDENCE_FILTERED_SUFFIX} found under {RESULTS_ROOT}"
    )

print(f"Found {len(patients)} patients:")
for patient in patients:
    print(f"  {patient.pid}  ({patient.confidence_filtered_path})")

# Glob per extension rather than iterdir()+filter, so the `_done.txt` marker file that
# main.py writes into each patient's plots/zoomed_in/ (see main.py) is never even
# visited, let alone opened as an image.
all_plots = sorted(
    p
    for patient in patients
    if patient.plot_dir.is_dir()
    for ext in PLOT_EXTENSIONS
    for p in patient.plot_dir.glob(f"*{ext}")
    if p.is_file()
)

CSV_FIELDS = ["pid", "chrom", "center", "filename", "decision", "note", "timestamp"]

if ANNOTATIONS_CSV.exists():
    existing = pd.read_csv(ANNOTATIONS_CSV)
    already_done = set(existing["filename"])
else:
    ANNOTATIONS_CSV.write_text(",".join(CSV_FIELDS) + "\n")
    already_done = set()

todo_plots = [p for p in all_plots if p.name not in already_done]

print(f"\n{len(all_plots)} plots found across {len(patients)} patients")
print(f"{len(already_done)} already annotated in {ANNOTATIONS_CSV}")
print(f"{len(todo_plots)} remaining to review")

In [ ]:
import threading
import time


def append_annotation(path: Path, decision: str, note: str):
    meta = parse_plot_filename(path)
    row = {
        "pid": meta["pid"],
        "chrom": meta["chrom"],
        "center": meta["center"],
        "filename": path.name,
        "decision": decision,
        "note": note,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    with open(ANNOTATIONS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        writer.writerow(row)


# --- Review UI state ---
state = {"index": 0, "busy": False, "last_click": -1.0}
_prefetch_cache = {}  # index -> rendered Image, filled by a background thread


def _prefetch(index):
    if 0 <= index < len(todo_plots) and index not in _prefetch_cache:
        try:
            _prefetch_cache[index] = render_plot_to_image(todo_plots[index])
        except Exception:
            pass  # surfaced again (and reported) when show_current() renders it directly


image_out = widgets.Output()
status_label = widgets.Label()
note_box = widgets.Text(
    placeholder="optional note / name",
    description="Note:",
    layout=widgets.Layout(width="500px"),
)
pass_btn = widgets.Button(description="Pass", button_style="success")
fail_btn = widgets.Button(description="Fail", button_style="danger")
skip_btn = widgets.Button(description="Skip (no annotation)")
prev_btn = widgets.Button(description="< Prev")
all_buttons = (prev_btn, pass_btn, fail_btn, skip_btn)


def show_current():
    image_out.clear_output(wait=True)
    note_box.value = ""
    if state["index"] >= len(todo_plots):
        status_label.value = "All plots reviewed."
        with image_out:
            print("Nothing left to review.")
        return

    idx = state["index"]
    path = todo_plots[idx]
    status_label.value = (
        f"{idx + 1} / {len(todo_plots)}  —  {path.name}  (rendering...)"
    )
    with image_out:
        print(
            f"Rendering {path.name} ..."
        )  # immediate feedback -- first render of a file can take a while

    try:
        img = _prefetch_cache.pop(idx, None) or render_plot_to_image(path)
        image_out.clear_output(wait=True)
        status_label.value = f"{idx + 1} / {len(todo_plots)}  —  {path.name}"
        with image_out:
            display(img)
    except Exception as e:
        image_out.clear_output(wait=True)
        with image_out:
            print(f"Could not render {path}: {e}")

    # kick off rendering the next plot in the background so it's ready by the time we get there
    threading.Thread(target=_prefetch, args=(idx + 1,), daemon=True).start()


def go_next(_=None):
    state["index"] += 1
    show_current()


# ipywidgets can fire a single click's on_click callback twice, as two separate
# kernel messages that are already both queued by the time the first is processed --
# so the fix has to be robust to the first one being slow (e.g. a cold PDF render).
# Measuring the debounce window from when the LAST action FINISHED (not from when it
# started) is what makes this work: a genuinely queued duplicate is always dispatched
# immediately once the first message's handler returns, regardless of how long that
# handler took, so the gap seen here stays near-zero. A real second click only ever
# lands well outside DOUBLE_CLICK_WINDOW because looking at a plot and deciding takes
# far longer than that. (An earlier version stamped last_click before calling fn(),
# which let duplicates through whenever the first click was slow to render.)
DOUBLE_CLICK_WINDOW = 0.4


def guarded(fn):
    def wrapped(_=None):
        now = time.monotonic()
        if state["busy"] or now - state["last_click"] < DOUBLE_CLICK_WINDOW:
            return
        state["busy"] = True
        for btn in all_buttons:
            btn.disabled = True
        try:
            fn()
        finally:
            state["busy"] = False
            state["last_click"] = time.monotonic()
            for btn in all_buttons:
                btn.disabled = False

    return wrapped


def _pass():
    append_annotation(todo_plots[state["index"]], "pass", note_box.value)
    go_next()


def _fail():
    append_annotation(todo_plots[state["index"]], "fail", note_box.value)
    go_next()


def _skip():
    go_next()


def _prev():
    if state["index"] > 0:
        state["index"] -= 1
        show_current()


on_pass = guarded(_pass)
on_fail = guarded(_fail)
on_skip = guarded(_skip)
on_prev = guarded(_prev)

pass_btn.on_click(on_pass)
fail_btn.on_click(on_fail)
skip_btn.on_click(on_skip)
prev_btn.on_click(on_prev)

controls = widgets.HBox([prev_btn, pass_btn, fail_btn, skip_btn])
display(widgets.VBox([status_label, image_out, note_box, controls]))
show_current()

## Write annotations back onto each patient's confidence-filtered table, summarize per PID

For each patient discovered above, left-joins your manual annotations onto their own
`*_extended_with_confidence_filtered.tsv` by `(chrom, insertion_site)` — the same
`(pid, chrom, center)` key used to name the plot files
(`make_bed_for_visualization.py` sets the plot's `region_center` to `insertion_site`
directly, so this is an exact match, not a binned/rounded one; every row in this table
already has a non-null `insertion_site`, since `filter_by_site_confidence.py` drops rows
without one). The manual decision/note become new columns (`manual_decision`,
`manual_note`) alongside all of the pipeline's existing filter/ratio/consensus columns.

Each patient's result is written to a sibling `*_confidence_filtered_annotated.tsv`
next to their original table — never overwriting the pipeline's own output, so a bad
annotation pass never destroys something you'd have to rerun the pipeline to get back.

Run this cell any time (including mid-review) — it always reflects whatever has been
annotated so far.

In [ ]:
if ANNOTATIONS_CSV.exists():
    # dtype=str on pid/chrom: pd.read_csv otherwise infers a numeric dtype whenever a
    # batch's annotated chroms happen to be all-digits (e.g. no X/Y reviewed yet), which
    # then fails to merge against the candidate table's string chrom column below.
    annotations = pd.read_csv(ANNOTATIONS_CSV, dtype={"pid": str, "chrom": str})
else:
    annotations = pd.DataFrame(columns=CSV_FIELDS)

annotations = annotations.rename(
    columns={"decision": "manual_decision", "note": "manual_note"}
)
annotations["center"] = annotations["center"].astype("Int64")

summary_rows = []
for patient in patients:
    candidates = pd.read_csv(
        patient.confidence_filtered_path, sep="\t", dtype={"chrom": str}
    )

    # region_center in the plot filename == insertion_site (see make_bed_for_visualization.py).
    plot_center = candidates["insertion_site"].round().astype("Int64")

    patient_annotations = annotations[annotations["pid"] == patient.pid]

    merged = candidates.copy()
    merged["_plot_center"] = plot_center
    merged = merged.merge(
        patient_annotations[
            ["chrom", "center", "manual_decision", "manual_note", "timestamp"]
        ],
        left_on=["chrom", "_plot_center"],
        right_on=["chrom", "center"],
        how="left",
    ).drop(columns=["_plot_center", "center"])

    out_path = patient.confidence_filtered_path.with_name(
        patient.confidence_filtered_path.stem + "_annotated.tsv"
    )
    merged.to_csv(out_path, sep="\t", index=False)

    n_annotated = int(merged["manual_decision"].notna().sum())
    n_passed = int((merged["manual_decision"] == "pass").sum())
    summary_rows.append(
        {
            "PID": patient.pid,
            "n_candidate_sites": len(merged),
            "n_annotated": n_annotated,
            "n_passed_insertion_sites": n_passed,
        }
    )
    print(
        f"Wrote {out_path}: {len(merged)} candidate sites, {n_annotated} annotated ({n_passed} passed)"
    )

summary = pd.DataFrame(summary_rows).sort_values("PID").reset_index(drop=True)
summary.to_csv(SUMMARY_CSV, index=False)
print(f"\nWrote {SUMMARY_CSV}")
summary